In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Dataset file unzip**

In [ ]:
import zipfile
import os

# ড্রাইভের পাথ (আপনার ফোল্ডার অনুযায়ী পরিবর্তন করুন)
zip_path = '/content/drive/MyDrive/Data Science/bus-stop-assess final.v2i.yolov8.zip'
extract_path = '/content/dataset'

if not os.path.exists(extract_path):
    os.makedirs(extract_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Drive থেকে ডেটাসেট কপি এবং আনজিপ সম্পন্ন হয়েছে!")

Drive থেকে ডেটাসেট কপি এবং আনজিপ সম্পন্ন হয়েছে!


৪. data.yaml ফাইল আপডেট করা

আনজিপ করার পর অবশ্যই data.yaml ফাইলের পাথ ঠিক করতে হবে যাতে মডেল ট্রেনিং শুরু করতে পারে:

In [ ]:
import yaml

yaml_path = '/content/dataset/data.yaml'

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Colab লোকাল পাথ সেট করা
data['train'] = '/content/dataset/train/images'
data['val'] = '/content/dataset/valid/images'
data['test'] = '/content/dataset/test/images'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

৫. ট্রেনিং শেষে "Weights" ড্রাইভে সেভ করা

ট্রেনিং শেষ হলে আপনার তৈরি হওয়া মডেল বা Weights (best.pt) যেন হারিয়ে না যায়, সেজন্য সেগুলো ড্রাইভে সেভ করে রাখুন:

In [ ]:
from ultralytics import YOLO

# মডেল লোড
model = YOLO('yolo11n.pt')

# ট্রেনিং শুরু (blur প্যারামিটারটি সরিয়ে দেওয়া হয়েছে কারণ এটি সরাসরি সাপোর্ট করে না)
model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    project='/content/drive/MyDrive/Thesis_Project/training_results',
    name='bus_stop_v1',

    # অনুমোদিত অগমেন্টেশন প্যারামিটারসমূহ
    scale=0.5,        # ইমেজ স্কেলিং
    perspective=0.0005, # পারসপেক্টিভ পরিবর্তন
    mosaic=1.0,       # ৪টি ইমেজ মিক্স করা (এটি ছোট ডেটাসেটের জন্য খুব শক্তিশালী)
    mixup=0.2,        # দুটি ইমেজ ব্লেন্ড করা (অ্যানিমেটেড ও রিয়েল মিক্স করতে সাহায্য করবে)
    degrees=10.0      # সামান্য রোটেশন
)

Ultralytics 8.4.31 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bus_stop_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspe

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a18c0371be0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

টেস্ট ডেটাসেটে মডেলের পারফরম্যান্স যাচাই (Testing)

আপনার কাছে ১১৮টি টেস্ট ইমেজ আছে যা মডেল আগে কখনো দেখেনি। এই ইমেজগুলোর ওপর প্রেডিকশন রান করে আপনার দেখা উচিত মডেলটি বাস্তবে কেমন কাজ করছে।

In [7]:
import zipfile
import os
from google.colab import drive
from ultralytics import YOLO

# ১. ড্রাইভ মাউন্ট (অলরেডি আছে, তাই সমস্যা নেই)
# drive.mount('/content/drive')

# ২. সঠিক পাথটি এখানে পেস্ট করুন (Copy path থেকে যেটা পেয়েছেন)
zip_path = '/content/drive/MyDrive/Data Science/bus-stop-assess final.v2i.yolov8.zip'
extract_path = '/content/dataset'

# ৩. আনজিপ করা
if not os.path.exists(extract_path):
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("Unzip সফল হয়েছে!")
    except FileNotFoundError:
        print("এরর: জিপ ফাইলটি এই পাথে পাওয়া যায়নি। দয়া করে 'Copy path' করে সঠিক পাথ দিন।")
else:
    print("ফোল্ডার অলরেডি আছে।")

# ৪. প্রেডিকশন রান করা (যদি আনজিপ সফল হয়)
if os.path.exists(extract_path):
    model_path = '/content/drive/MyDrive/Thesis_Project/training_results/bus_stop_v1/weights/best.pt'
    model = YOLO(model_path)

    test_images_path = '/content/dataset/test/images'

    results = model.predict(
        source=test_images_path,
        save=True,
        conf=0.25,
        project='/content/drive/MyDrive/Thesis_Project',
        name='test_results'
    )
    print("প্রেডিকশন শেষ!")

Unzip সফল হয়েছে!

image 1/118 /content/dataset/test/images/1F5kE4ER0UZz8BDchBzLcA_jpg.rf.0c300b9c20a106c0a3a8f41cfb34a295.jpg: 640x640 1 seating, 1 shelter, 1 trash can, 10.4ms
image 2/118 /content/dataset/test/images/210170_1_jpg.rf.6f4e4fedb92493635b2ff2f16010db06.jpg: 640x640 1 sign, 1 trash can, 11.0ms
image 3/118 /content/dataset/test/images/211401_0_jpg.rf.a4671744c3127b679eede33211c5d91b.jpg: 640x640 (no detections), 10.3ms
image 4/118 /content/dataset/test/images/211997_0_jpg.rf.2bb3ca346f3f71378b912889146b78d9.jpg: 640x640 1 seating, 1 trash can, 10.2ms
image 5/118 /content/dataset/test/images/212079_0_jpg.rf.4f75a99bfddaa3e3f2a0eda98319c7ec.jpg: 640x640 1 route info, 1 seating, 1 sign, 1 trash can, 11.0ms
image 6/118 /content/dataset/test/images/212106_1_jpg.rf.b7db26c24c34b7e3e3d9fa99edb912dd.jpg: 640x640 1 sign, 10.2ms
image 7/118 /content/dataset/test/images/212144_1_jpg.rf.32a2e1d323f5808ee9b4c72e4ee37af1.jpg: 640x640 1 route info, 1 schedule, 1 seating, 1 shelter, 1 sign

যদি আপনি নিজে কোড দিয়ে একবার চেক করতে চান (Validation Report)

আপনি যদি চান আপনার ১১৮টি টেস্ট ইমেজের ওপর ভিত্তি করে একটি ফ্রেশ রিপোর্ট দেখতে, তবে এই ছোট কোডটি Colab-এ রান করুন:

In [8]:
from ultralytics import YOLO

# ১. ড্রাইভ থেকে মডেল লোড করুন
model_path = '/content/drive/MyDrive/Thesis_Project/training_results/bus_stop_v1/weights/best.pt'
model = YOLO(model_path)

# ২. ভ্যালিডেশন রান করুন (এটি আপনার টেস্ট ডেটাসেটের ওপর সব গ্রাফ দেখাবে)
metrics = model.val(data='/content/dataset/data.yaml', split='test')

# ৩. রেজাল্ট প্রিন্ট করা
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1573.1±429.9 MB/s, size: 92.9 KB)
val: Scanning /content/dataset/test/labels... 118 images, 8 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 118/118 1.1Kit/s 0.1s
val: New cache created: /content/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.5it/s 5.2s
                   all        118        380      0.946      0.791       0.88      0.609
            route info         30         30      0.898      0.633      0.742        0.4
              schedule         33         33      0.945      0.848       0.94       0.71
               seating         71         72      0.892      0.804      0.886       0.58
               shelter         61         61      0.991      0.934       0.98      0.8